<a href="https://colab.research.google.com/github/mxls34/AdvanceDatabase/blob/main/Chapter3_HW_Joins_and_Join_Algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

รหัสนักศึกษา:6706021612143

ชื่อ-นามสกุล:นาย ณัฐกรณ์ มะลิซ้อน


# การบ้าน Chapter 3: Joins & Join Algorithms

**สถานการณ์:** โรงเรียนสอนภาษาแห่งหนึ่งมีข้อมูล 3 ตาราง
- `student` (นักเรียน)
- `course` (คอร์สเรียน)
- `enrollment` (การลงทะเบียนเรียน)

ผู้บริหารต้องการให้คุณช่วยวิเคราะห์ข้อมูลเพื่อทำความเข้าใจภาพรวมของโรงเรียน
โดยใช้เทคนิค JOIN

## Setup — DuckDB

In [12]:
!pip install duckdb --quiet


Create DB

In [13]:
import duckdb

con = duckdb.connect(database=':memory:')

def run(sql):
    try:
        return con.execute(sql).df()
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

con.execute("CREATE TABLE student (student_id INTEGER, name VARCHAR, city VARCHAR)")
con.executemany("INSERT INTO student VALUES (?,?,?)", [
    (1, 'ปกรณ์', 'กรุงเทพฯ'), (2, 'ณัฐวุฒิ', 'กรุงเทพฯ'), (3, 'ชนากานต์', 'กรุงเทพฯ'),
    (4, 'ธีรภัทร', 'เชียงใหม่'), (5, 'พิมพ์ชนก', 'เชียงใหม่'),
    (6, 'กิตติศักดิ์', 'ขอนแก่น'), (7, 'ศศิธร', 'ขอนแก่น'),
    (8, 'อนุชา', 'ภูเก็ต'), (9, 'วรรณิศา', 'ภูเก็ต'),
    (10, 'ปัญญา', 'กรุงเทพฯ'), (11, 'รุ่งนภา', 'เชียงใหม่'),
    (12, 'เอกชัย', 'สงขลา'), (13, 'สิริกร', 'นครราชสีมา'), (14, 'ธนพล', 'อุดรธานี'),
    (15, 'มนัสวี', 'ขอนแก่น'),
])

con.execute("CREATE TABLE course (course_id INTEGER, course_name VARCHAR, teacher VARCHAR, fee DECIMAL(10,2))")
con.executemany("INSERT INTO course VALUES (?,?,?,?)", [
    (201, 'ภาษาอังกฤษพื้นฐาน', 'ครูสมใจ', 3500.00),
    (202, 'ภาษาอังกฤษเพื่อธุรกิจ', 'ครูสมใจ', 4500.00),
    (203, 'ภาษาญี่ปุ่น N5', 'ครูยูกิ', 4000.00),
    (204, 'ภาษาญี่ปุ่น N4', 'ครูยูกิ', 4200.00),
    (205, 'ภาษาจีนพื้นฐาน', 'ครูเหมย', 3800.00),
    (206, 'ภาษาเกาหลีพื้นฐาน', 'ครูจีอึน', 3900.00),
    (207, 'ภาษาฝรั่งเศสพื้นฐาน', 'ครูโซฟี', 4100.00),
    (208, 'ภาษาเยอรมันพื้นฐาน', 'ครูฮันส์', 4300.00),
])

con.execute("""CREATE TABLE enrollment (
    enrollment_id INTEGER, student_id INTEGER, course_id INTEGER,
    enroll_date DATE, status VARCHAR, score INTEGER
)""")
con.executemany("INSERT INTO enrollment VALUES (?,?,?,?,?,?)", [
    (1, 1, 201, '2024-01-10', 'completed', 85), (2, 1, 203, '2024-02-15', 'active', None),
    (3, 2, 201, '2024-01-12', 'completed', 78), (4, 2, 202, '2024-04-01', 'active', None),
    (5, 3, 205, '2024-01-20', 'completed', 92), (6, 4, 203, '2024-01-25', 'completed', 88),
    (7, 4, 204, '2024-05-10', 'active', None), (8, 5, 206, '2024-02-05', 'dropped', None),
    (9, 6, 201, '2024-03-01', 'completed', 74), (10, 6, 205, '2024-06-15', 'active', None),
    (11, 7, 202, '2024-02-20', 'completed', 81), (12, 8, 203, '2024-01-18', 'completed', 90),
    (13, 8, 206, '2024-04-22', 'dropped', None), (14, 9, 201, '2024-03-10', 'completed', 69),
    (15, 10, 205, '2024-02-01', 'completed', 95), (16, 10, 201, '2024-07-01', 'active', None),
    (17, 11, 206, '2024-01-30', 'completed', 83), (18, 15, 203, '2024-03-15', 'completed', 91),
    (19, 15, 204, '2024-06-01', 'active', None), (20, 2, 205, '2024-07-10', 'active', None),
    (21, 9, 202, '2024-05-05', 'completed', 87), (22, 6, 202, '2024-07-15', 'dropped', None),
])

print("Setup complete — พร้อมเริ่มทำการบ้าน")

Setup complete — พร้อมเริ่มทำการบ้าน


แสดงข้อมูลในตาราง student ออกมาดู 5 records แรก

In [ ]:
# เขียนโค้ดแสดงข้อมูลในตาราง student ตรงนี้
run("SELECT * FROM student LIMIT 5")

,student_id,name,city
0,1,ปกรณ์,กรุงเทพฯ
1,2,ณัฐวุฒิ,กรุงเทพฯ
2,3,ชนากานต์,กรุงเทพฯ
3,4,ธีรภัทร,เชียงใหม่
4,5,พิมพ์ชนก,เชียงใหม่


In [ ]:
# ตัวอย่างผลลัพธ์

,student_id,name,city
0,1,ปกรณ์,กรุงเทพฯ
1,2,ณัฐวุฒิ,กรุงเทพฯ
2,3,ชนากานต์,กรุงเทพฯ
3,4,ธีรภัทร,เชียงใหม่
4,5,พิมพ์ชนก,เชียงใหม่


แสดงข้อมูลในตาราง course ออกมาดู 5 records แรก

In [ ]:
# เขียนโค้ดแสดงข้อมูลในตาราง course ตรงนี้
run("SELECT * FROM course LIMIT 5")

,course_id,course_name,teacher,fee
0,201,ภาษาอังกฤษพื้นฐาน,ครูสมใจ,3500.0
1,202,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ,4500.0
2,203,ภาษาญี่ปุ่น N5,ครูยูกิ,4000.0
3,204,ภาษาญี่ปุ่น N4,ครูยูกิ,4200.0
4,205,ภาษาจีนพื้นฐาน,ครูเหมย,3800.0


In [ ]:
# ตัวอย่างผลลัพธ์

,course_id,course_name,teacher,fee
0,201,ภาษาอังกฤษพื้นฐาน,ครูสมใจ,3500.0
1,202,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ,4500.0
2,203,ภาษาญี่ปุ่น N5,ครูยูกิ,4000.0
3,204,ภาษาญี่ปุ่น N4,ครูยูกิ,4200.0
4,205,ภาษาจีนพื้นฐาน,ครูเหมย,3800.0


แสดงข้อมูลในตาราง enrollment ออกมาดู 5 records แรก

In [ ]:
# เขียนโค้ดแสดงข้อมูลในตาราง enrollment ตรงนี้
run("SELECT * FROM enrollment LIMIT 5")

,enrollment_id,student_id,course_id,enroll_date,status,score
0,1,1,201,2024-01-10,completed,85
1,2,1,203,2024-02-15,active,<NA>
2,3,2,201,2024-01-12,completed,78
3,4,2,202,2024-04-01,active,<NA>
4,5,3,205,2024-01-20,completed,92


In [ ]:
# ตัวอย่างผลลัพธ์

,enrollment_id,student_id,course_id,enroll_date,status,score
0,1,1,201,2024-01-10,completed,85
1,2,1,203,2024-02-15,active,<NA>
2,3,2,201,2024-01-12,completed,78
3,4,2,202,2024-04-01,active,<NA>
4,5,3,205,2024-01-20,completed,92


---
# Part 1: JOIN พื้นฐาน


In [ ]:
run("SHOW TABLES")

,name
0,course
1,enrollment
2,student


In [ ]:
run("DESCRIBE course")

,column_name,column_type,null,key,default,extra
0,course_id,INTEGER,YES,None,None,None
1,course_name,VARCHAR,YES,None,None,None
2,teacher,VARCHAR,YES,None,None,None
3,fee,"DECIMAL(10,2)",YES,None,None,None


In [ ]:
run("DESCRIBE enrollment")

,column_name,column_type,null,key,default,extra
0,enrollment_id,INTEGER,YES,None,None,None
1,student_id,INTEGER,YES,None,None,None
2,course_id,INTEGER,YES,None,None,None
3,enroll_date,DATE,YES,None,None,None
4,status,VARCHAR,YES,None,None,None
5,score,INTEGER,YES,None,None,None


**1.1** เชื่อม `enrollment` กับ `student` แสดงชื่อนักเรียนที่ลงทะเบียนแต่ละรายการ พร้อม course_id และ status

In [ ]:
# เขียนโค้ดของข้อ 1.1 ตรงนี้
run("""
  SELECT s.name, e.course_id, e.status FROM enrollment e JOIN student s ON e.student_id = s.student_id
""")

,name,course_id,status
0,ปกรณ์,201,completed
1,ปกรณ์,203,active
2,ณัฐวุฒิ,201,completed
3,ณัฐวุฒิ,202,active
4,ชนากานต์,205,completed
5,ธีรภัทร,203,completed
6,ธีรภัทร,204,active
7,พิมพ์ชนก,206,dropped
8,กิตติศักดิ์,201,completed
9,กิตติศักดิ์,205,active


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1.1

,name,course_id,status
0,ปกรณ์,201,completed
1,ปกรณ์,203,active
2,ณัฐวุฒิ,201,completed
3,ณัฐวุฒิ,202,active
4,ชนากานต์,205,completed
5,ธีรภัทร,203,completed
6,ธีรภัทร,204,active
7,พิมพ์ชนก,206,dropped
8,กิตติศักดิ์,201,completed
9,กิตติศักดิ์,205,active


**1.2** เชื่อม `enrollment` กับ `course` แสดงชื่อคอร์สที่ถูกลงทะเบียน พร้อมชื่อครูผู้สอน

In [ ]:
# เขียนโค้ดของข้อ 1.2 ตรงนี้
run("""
  SELECT enrollment_id, course_name, teacher FROM enrollment e JOIN course c ON e.course_id = c.course_id
""")

,enrollment_id,course_name,teacher
0,1,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
1,2,ภาษาญี่ปุ่น N5,ครูยูกิ
2,3,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
3,4,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ
4,5,ภาษาจีนพื้นฐาน,ครูเหมย
5,6,ภาษาญี่ปุ่น N5,ครูยูกิ
6,7,ภาษาญี่ปุ่น N4,ครูยูกิ
7,8,ภาษาเกาหลีพื้นฐาน,ครูจีอึน
8,9,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
9,10,ภาษาจีนพื้นฐาน,ครูเหมย


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1.2

,enrollment_id,course_name,teacher
0,1,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
1,2,ภาษาญี่ปุ่น N5,ครูยูกิ
2,3,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
3,4,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ
4,5,ภาษาจีนพื้นฐาน,ครูเหมย
5,6,ภาษาญี่ปุ่น N5,ครูยูกิ
6,7,ภาษาญี่ปุ่น N4,ครูยูกิ
7,8,ภาษาเกาหลีพื้นฐาน,ครูจีอึน
8,9,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
9,10,ภาษาจีนพื้นฐาน,ครูเหมย


**1.3** JOIN สามตาราง (`student` + `course` + `enrollment`) แสดง "ใครเรียนคอร์สอะไร กับครูคนไหน"

In [ ]:
# เขียนโค้ดของข้อ 1.3 ตรงนี้
run("""
    SELECT name, course_name, teacher FROM student s JOIN enrollment e ON s.student_id = e.student_id JOIN course c ON e.course_id = c.course_id

""")

,name,course_name,teacher
0,ปกรณ์,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
1,ปกรณ์,ภาษาญี่ปุ่น N5,ครูยูกิ
2,ณัฐวุฒิ,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
3,ณัฐวุฒิ,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ
4,ชนากานต์,ภาษาจีนพื้นฐาน,ครูเหมย
5,ธีรภัทร,ภาษาญี่ปุ่น N5,ครูยูกิ
6,ธีรภัทร,ภาษาญี่ปุ่น N4,ครูยูกิ
7,พิมพ์ชนก,ภาษาเกาหลีพื้นฐาน,ครูจีอึน
8,กิตติศักดิ์,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
9,กิตติศักดิ์,ภาษาจีนพื้นฐาน,ครูเหมย


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1.3

,name,course_name,teacher
0,ปกรณ์,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
1,ปกรณ์,ภาษาญี่ปุ่น N5,ครูยูกิ
2,ณัฐวุฒิ,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
3,ณัฐวุฒิ,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ
4,ชนากานต์,ภาษาจีนพื้นฐาน,ครูเหมย
5,ธีรภัทร,ภาษาญี่ปุ่น N5,ครูยูกิ
6,ธีรภัทร,ภาษาญี่ปุ่น N4,ครูยูกิ
7,พิมพ์ชนก,ภาษาเกาหลีพื้นฐาน,ครูจีอึน
8,กิตติศักดิ์,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
9,กิตติศักดิ์,ภาษาจีนพื้นฐาน,ครูเหมย


**1.4** หารายชื่อนักเรียนที่ลงทะเบียนคอร์สของ "ครูสมใจ" เท่านั้น (JOIN + WHERE)

In [ ]:
run("DESCRIBE course")

,column_name,column_type,null,key,default,extra
0,course_id,INTEGER,YES,None,None,None
1,course_name,VARCHAR,YES,None,None,None
2,teacher,VARCHAR,YES,None,None,None
3,fee,"DECIMAL(10,2)",YES,None,None,None


In [ ]:
run("DESCRIBE student")

,column_name,column_type,null,key,default,extra
0,student_id,INTEGER,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,city,VARCHAR,YES,None,None,None


In [ ]:
# เขียนโค้ดของข้อ 1.4 ตรงนี้
run("""
  SELECT s.name, c.course_name, c.teacher FROM student s JOIN enrollment e ON s.student_id = e.student_id JOIN course c ON e.course_id = c.course_id WHERE teacher IN('ครูสมใจ')
""")

,name,course_name,teacher
0,ปกรณ์,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
1,ณัฐวุฒิ,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ
2,กิตติศักดิ์,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ
3,ศศิธร,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ
4,วรรณิศา,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ
5,ปัญญา,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
6,ณัฐวุฒิ,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
7,กิตติศักดิ์,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
8,วรรณิศา,ภาษาอังกฤษพื้นฐาน,ครูสมใจ


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1.4

,name,course_name,teacher
0,ปกรณ์,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
1,ณัฐวุฒิ,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ
2,กิตติศักดิ์,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ
3,ศศิธร,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ
4,วรรณิศา,ภาษาอังกฤษเพื่อธุรกิจ,ครูสมใจ
5,ปัญญา,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
6,ณัฐวุฒิ,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
7,กิตติศักดิ์,ภาษาอังกฤษพื้นฐาน,ครูสมใจ
8,วรรณิศา,ภาษาอังกฤษพื้นฐาน,ครูสมใจ


**1.5** หารายการที่สถานะเป็น `'completed'` พร้อมคะแนน (score) เรียงจากคะแนนมากไปน้อย

In [ ]:
# เขียนโค้ดของข้อ 1.5 ตรงนี้
run("""
  SELECT s.name, c.course_name, e.score, e.status FROM student s JOIN enrollment e ON s.student_id = e.student_id JOIN course c ON e.course_id = c.course_id WHERE status IN('completed') ORDER BY score DESC
""")

,name,course_name,score,status
0,ปัญญา,ภาษาจีนพื้นฐาน,95,completed
1,ชนากานต์,ภาษาจีนพื้นฐาน,92,completed
2,มนัสวี,ภาษาญี่ปุ่น N5,91,completed
3,อนุชา,ภาษาญี่ปุ่น N5,90,completed
4,ธีรภัทร,ภาษาญี่ปุ่น N5,88,completed
5,วรรณิศา,ภาษาอังกฤษเพื่อธุรกิจ,87,completed
6,ปกรณ์,ภาษาอังกฤษพื้นฐาน,85,completed
7,รุ่งนภา,ภาษาเกาหลีพื้นฐาน,83,completed
8,ศศิธร,ภาษาอังกฤษเพื่อธุรกิจ,81,completed
9,ณัฐวุฒิ,ภาษาอังกฤษพื้นฐาน,78,completed


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1.5

,name,course_name,score,status
0,ปัญญา,ภาษาจีนพื้นฐาน,95,completed
1,ชนากานต์,ภาษาจีนพื้นฐาน,92,completed
2,มนัสวี,ภาษาญี่ปุ่น N5,91,completed
3,อนุชา,ภาษาญี่ปุ่น N5,90,completed
4,ธีรภัทร,ภาษาญี่ปุ่น N5,88,completed
5,วรรณิศา,ภาษาอังกฤษเพื่อธุรกิจ,87,completed
6,ปกรณ์,ภาษาอังกฤษพื้นฐาน,85,completed
7,รุ่งนภา,ภาษาเกาหลีพื้นฐาน,83,completed
8,ศศิธร,ภาษาอังกฤษเพื่อธุรกิจ,81,completed
9,ณัฐวุฒิ,ภาษาอังกฤษพื้นฐาน,78,completed


---
# Part 2: OUTER JOIN — สำรวจข้อมูลที่ขาดหาย (EDA)


**2.1** หานักเรียนที่ **ไม่เคยลงทะเบียนเรียน** คอร์สใดเลย

In [22]:
# เขียนโค้ดของข้อ 2.1 ตรงนี้
run("""
  SELECT s.name, s.city, e.enrollment_id FROM student s
  FULL JOIN enrollment e ON s.student_id = e.student_id
  FULL JOIN course c ON e.course_id = c.course_id WHERE s.student_id NOT NULL AND enrollment_id IS NULL
""")

,name,city,enrollment_id
0,เอกชัย,สงขลา,<NA>
1,สิริกร,นครราชสีมา,<NA>
2,ธนพล,อุดรธานี,<NA>


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 2.1

,name,city,enrollment_id
0,เอกชัย,สงขลา,<NA>
1,สิริกร,นครราชสีมา,<NA>
2,ธนพล,อุดรธานี,<NA>


**2.2** หาคอร์สที่ **ไม่เคยมีนักเรียนลงทะเบียน** เลยสักคน

In [25]:
# เขียนโค้ดของข้อ 2.2 ตรงนี้
run("""
  SELECT c.course_id, c.course_name, c.teacher, e.enrollment_id FROM student s
  FULL JOIN enrollment e ON s.student_id = e.student_id
  FULL JOIN course c ON e.course_id = c.course_id WHERE enrollment_id IS NULL AND c.course_id NOT NULL
""")

,course_id,course_name,teacher,enrollment_id
0,207,ภาษาฝรั่งเศสพื้นฐาน,ครูโซฟี,<NA>
1,208,ภาษาเยอรมันพื้นฐาน,ครูฮันส์,<NA>


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 2.2

,course_id,course_name,teacher,enrollment_id
0,207,ภาษาฝรั่งเศสพื้นฐาน,ครูโซฟี,<NA>
1,208,ภาษาเยอรมันพื้นฐาน,ครูฮันส์,<NA>


**2.3** เทียบจำนวนแถวจาก `INNER JOIN` กับ `LEFT JOIN` ของ `student` กับ `enrollment`
แล้วอธิบายว่าทำไมตัวเลขถึงต่างกัน

In [30]:
# เขียนโค้ดของข้อ 2.3 ตรงนี้
inner_n = con.execute("SELECT COUNT(*) FROM student s INNER JOIN enrollment e ON s.student_id = e.student_id").fetchone()[0]
left_n  = con.execute("SELECT COUNT(*) FROM student s LEFT JOIN enrollment e ON s.student_id = e.student_id").fetchone()[0]
print("INNER JOIN:", inner_n, " | LEFT JOIN:", left_n)

INNER JOIN: 22  | LEFT JOIN: 25


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 2.3

INNER JOIN: 22  | LEFT JOIN: 25


_เขียนคำอธิบายตรงนี้:_ ...

**2.4** ใช้ `LEFT JOIN` หานักเรียนทุกคน พร้อมคะแนนสอบ (score) — รวมนักเรียนที่ยังไม่มีคะแนนด้วย

In [35]:
# เขียนโค้ดของข้อ 2.4 ตรงนี้
run("""

""")

BinderException: Binder Error: Referenced table "e" not found!
Candidate tables: "s"

LINE 3:   LEFT JOIN course c ON e.course_id = c.course_id
                                ^


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 2.4

,name,course_id,status,score
0,ปกรณ์,201,completed,85
1,ปกรณ์,203,active,<NA>
2,ณัฐวุฒิ,201,completed,78
3,ณัฐวุฒิ,202,active,<NA>
4,ณัฐวุฒิ,205,active,<NA>
5,ชนากานต์,205,completed,92
6,ธีรภัทร,203,completed,88
7,ธีรภัทร,204,active,<NA>
8,พิมพ์ชนก,206,dropped,<NA>
9,กิตติศักดิ์,201,completed,74


---
# Part 3: JOIN ขั้นสูง


**3.1** Self-join: หาคู่นักเรียนที่อยู่**เมืองเดียวกัน** และเคยลงทะเบียน**คอร์สเดียวกัน**
(ต้องใช้ Multi-condition join ร่วมกับ self-join)

In [ ]:
# เขียนโค้ดของข้อ 3.1 ตรงนี้
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 3.1

,student1,student2,city,course_name
0,ณัฐวุฒิ,ปัญญา,กรุงเทพฯ,ภาษาอังกฤษพื้นฐาน
1,ปกรณ์,ณัฐวุฒิ,กรุงเทพฯ,ภาษาอังกฤษพื้นฐาน
2,กิตติศักดิ์,ศศิธร,ขอนแก่น,ภาษาอังกฤษเพื่อธุรกิจ
3,ชนากานต์,ปัญญา,กรุงเทพฯ,ภาษาจีนพื้นฐาน
4,ปกรณ์,ปัญญา,กรุงเทพฯ,ภาษาอังกฤษพื้นฐาน
5,พิมพ์ชนก,รุ่งนภา,เชียงใหม่,ภาษาเกาหลีพื้นฐาน
6,ณัฐวุฒิ,ชนากานต์,กรุงเทพฯ,ภาษาจีนพื้นฐาน
7,ณัฐวุฒิ,ปัญญา,กรุงเทพฯ,ภาษาจีนพื้นฐาน


**3.2** Non-equi join: สร้างตาราง `quarter` แบ่งปี 2024 เป็น 4 ไตรมาส
(Q1: ม.ค.-มี.ค., Q2: เม.ย.-มิ.ย., Q3: ก.ค.-ก.ย., Q4: ต.ค.-ธ.ค.) แล้วนับจำนวนการลงทะเบียนในแต่ละไตรมาส

In [ ]:
# โค้ดของข้อ 3.2 สร้างตาราง quarter
con.execute("CREATE TABLE quarter (q_name VARCHAR, start_date DATE, end_date DATE)")
con.executemany("INSERT INTO quarter VALUES (?,?,?)", [
    ('Q1', '2024-01-01', '2024-03-31'), ('Q2', '2024-04-01', '2024-06-30'),
    ('Q3', '2024-07-01', '2024-09-30'), ('Q4', '2024-10-01', '2024-12-31'),
])

In [ ]:
# เขียนโค้ดของข้อ 3.2 ตรงนี้

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 3.2

,q_name,enroll_count
0,Q1,13
1,Q2,6
2,Q3,3


**3.3** หาคอร์สที่มีคะแนนเฉลี่ย (เฉพาะคนที่ `status = 'completed'`) **สูงที่สุด**
(ต้อง JOIN + WHERE + GROUP BY ในโจทย์เดียว)

In [ ]:
# เขียนโค้ดของข้อ 3.3 ตรงนี้
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 3.3

,course_name,avg_score,n_completed
0,ภาษาจีนพื้นฐาน,93.5,2


จบแบบฝึกหัด Chapter 3